In [53]:
!pip install pypdf sentence-transformers faiss-cpu transformers
from google.colab import files
uploaded = files.upload()

Saving Electrical Dataset.pdf to Electrical Dataset (4).pdf


In [54]:
from pypdf import PdfReader

documents = []

for pdf_name in uploaded.keys():
    reader = PdfReader(pdf_name)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            documents.append({
                "text": text.replace("\n", " "),
                "source": pdf_name,
                "page": page_num + 1
            })

print("Total pages loaded:", len(documents))

Total pages loaded: 2


In [55]:
def chunk_documents(documents, chunk_size=120, overlap=30):
    chunks = []

    for doc in documents:
        words = doc["text"].split()

        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])

            if len(chunk) > 20:
                chunks.append({
                    "text": chunk,
                    "source": doc["source"],
                    "page": doc["page"]
                })

    return chunks


chunks = chunk_documents(documents)

print("Chunks:", len(chunks))

Chunks: 6


In [56]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model_embed = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]

embeddings = model_embed.encode(texts, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS ready:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS ready: 6


In [57]:
!pip install --upgrade transformers accelerate

In [58]:
def retrieve(query, k=5):
    q_emb = model_embed.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)

    scores, indices = index.search(q_emb, k)

    results = []
    for score, i in zip(scores[0], indices[0]):
        if i < len(chunks):
            results.append(chunks[i])

    return results

In [59]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [75]:
def ask_pdf(question):
    retrieved = retrieve(question, k=8)

    context = retrieved[0]["text"][:400]

    prompt = f"""
You are a strict science dictionary.

RULES:
- Give ONLY 1–2 sentences
- Do NOT repeat yourself
- Do NOT add extra explanations
- If not found, say "I don't know"

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, retrieved

In [76]:
while True:
    q = input("\nAsk a question (type exit): ")

    if q.lower() == "exit":
        break

    answer, sources = ask_pdf(q)

    print("\nAnswer:")
    print(textwrap.fill(answer, 80))

    print("\nSources:")
    for s in sources:
        print(f"- {s['source']} (Page {s['page']})")


Ask a question (type exit): what is voltage?

Answer:
Voltage is the electric potential difference between two points in a circuit. It
is measured in volts (V).

Sources:
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 2)

Ask a question (type exit): what is electrical resistance?

Answer:
opposition to the flow of electric current in a material

Sources:
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 1)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 2)
- Electrical Dataset (4).pdf (Page 2)

Ask a question (type exit): exit
